# Phase 7B — Computer Vision: Pillow & OpenCV

**Theory:** Image representation (pixels, channels, color spaces), basic image processing, edge detection, morphological operations.

**Install:** `pip install pillow opencv-python`

**PIL (Pillow):** High-level image handling — loading, saving, resizing, basic transforms.
**OpenCV:** Advanced image processing — filters, edge detection, geometric transforms, video.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

PIL_AVAILABLE = False
CV2_AVAILABLE = False

try:
    from PIL import Image, ImageFilter, ImageEnhance, ImageDraw

    PIL_AVAILABLE = True
    print("Pillow ready")
except ImportError:
    print("Install: pip install pillow")

try:
    import cv2

    CV2_AVAILABLE = True
    print(f"OpenCV ready: {cv2.__version__}")
except ImportError:
    print("Install: pip install opencv-python")

---
## 1. Image Fundamentals — Pixels and Arrays

A digital image is a 3D numpy array: `(height, width, channels)`
- Grayscale: `(H, W)` — one channel, values 0–255
- RGB: `(H, W, 3)` — three channels (Red, Green, Blue)
- RGBA: `(H, W, 4)` — with transparency channel

In [ ]:
# Create test images from scratch with NumPy

# Simple gradient image
gradient = np.zeros((100, 100, 3), dtype=np.uint8)
for i in range(100):
    gradient[i, :, 0] = int(i * 2.55)  # Red increases top to bottom
    gradient[:, i, 2] = int(i * 2.55)  # Blue increases left to right

# Checkerboard
checker = np.zeros((100, 100), dtype=np.uint8)
for i in range(100):
    for j in range(100):
        if (i // 10 + j // 10) % 2 == 0:
            checker[i, j] = 255

# Noise image
noise = np.random.randint(0, 256, (100, 100, 3), dtype=np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(gradient)
axes[0].set_title(f"RGB Gradient\nshape={gradient.shape}")
axes[0].axis("off")

axes[1].imshow(checker, cmap="gray")
axes[1].set_title(f"Grayscale Checkerboard\nshape={checker.shape}")
axes[1].axis("off")

axes[2].imshow(noise)
axes[2].set_title(f"Random Noise\nshape={noise.shape}")
axes[2].axis("off")

plt.tight_layout()
plt.show()

print(f"Pixel at (0,0) of gradient: {gradient[0, 0]}  (R=0, G=0, B=0)")
print(f"Pixel at (99,99) of gradient: {gradient[99, 99]}  (R=252, G=0, B=252)")

---
## 2. Pillow — Image Loading and Transforms

In [ ]:
if PIL_AVAILABLE:
    # Create an image from the numpy gradient array
    img = Image.fromarray(gradient)

    print(f"Image size   : {img.size}  (width, height)")
    print(f"Image mode   : {img.mode}")
    print(f"numpy shape  : {np.array(img).shape}  (height, width, channels)")

    # Common transforms
    transforms = {
        "Original": img,
        "Resize (50x50)": img.resize((50, 50)),
        "Rotate 45°": img.rotate(45, expand=True),
        "Flip Horizontal": img.transpose(Image.FLIP_LEFT_RIGHT),
        "Grayscale": img.convert("L"),
        "Blur": img.filter(ImageFilter.GaussianBlur(radius=3)),
    }

    fig, axes = plt.subplots(2, 3, figsize=(12, 7))
    for ax, (name, transformed) in zip(axes.flat, transforms.items()):
        if transformed.mode == "L":
            ax.imshow(np.array(transformed), cmap="gray")
        else:
            ax.imshow(np.array(transformed))
        ax.set_title(f"{name}\n{transformed.size}")
        ax.axis("off")

    plt.suptitle("Pillow Image Transforms", fontsize=13)
    plt.tight_layout()
    plt.show()

---
## 3. OpenCV — Advanced Image Processing

In [ ]:
if CV2_AVAILABLE:
    # OpenCV uses BGR (not RGB) — important!
    # Convert our gradient to BGR for OpenCV
    img_bgr = cv2.cvtColor(gradient, cv2.COLOR_RGB2BGR)

    # Grayscale conversion
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # Gaussian blur (reduce noise before edge detection)
    blurred = cv2.GaussianBlur(gray, ksize=(5, 5), sigmaX=1)

    # Canny edge detection — two thresholds
    edges = cv2.Canny(blurred, threshold1=50, threshold2=150)

    # Sobel gradients (X and Y direction)
    sobelx = cv2.Sobel(gray, cv2.CV_64F, dx=1, dy=0, ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_64F, dx=0, dy=1, ksize=3)
    sobel_combined = np.sqrt(sobelx**2 + sobely**2)

    # Thresholding
    _, thresh_binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
    _, thresh_otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    titles = [
        "Original (gray)",
        "Gaussian Blur",
        "Canny Edges",
        "Sobel Combined",
        "Binary Thresh",
        "Otsu Thresh",
    ]
    images = [
        gray,
        blurred,
        edges,
        sobel_combined.astype(np.uint8),
        thresh_binary,
        thresh_otsu,
    ]

    fig, axes = plt.subplots(2, 3, figsize=(12, 7))
    for ax, title, img_show in zip(axes.flat, titles, images):
        ax.imshow(img_show, cmap="gray")
        ax.set_title(title)
        ax.axis("off")

    plt.suptitle("OpenCV Image Processing", fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
if CV2_AVAILABLE:
    # Color space analysis
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    fig, axes = plt.subplots(2, 4, figsize=(14, 6))

    # RGB channels
    channel_names = ["Red", "Green", "Blue"]
    axes[0, 0].imshow(gradient)
    axes[0, 0].set_title("Original RGB")
    axes[0, 0].axis("off")

    for i, (name, color) in enumerate(zip(channel_names, ["Reds", "Greens", "Blues"])):
        axes[0, i + 1].imshow(gradient[:, :, i], cmap=color)
        axes[0, i + 1].set_title(f"RGB: {name} channel")
        axes[0, i + 1].axis("off")

    # HSV channels
    hsv_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    hsv_channel_names = ["Hue", "Saturation", "Value (Brightness)"]
    axes[1, 0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    axes[1, 0].set_title("Original (HSV space)")
    axes[1, 0].axis("off")

    for i, name in enumerate(hsv_channel_names):
        axes[1, i + 1].imshow(hsv_rgb[:, :, i], cmap="gray")
        axes[1, i + 1].set_title(f"HSV: {name}")
        axes[1, i + 1].axis("off")

    plt.suptitle("Color Space Analysis: RGB vs HSV", fontsize=13)
    plt.tight_layout()
    plt.show()

---
## Summary

| Task | Pillow | OpenCV |
|------|--------|--------|
| Load image | `Image.open(path)` | `cv2.imread(path)` |
| Save image | `img.save(path)` | `cv2.imwrite(path, img)` |
| Resize | `img.resize((w,h))` | `cv2.resize(img, (w,h))` |
| Convert to numpy | `np.array(img)` | already numpy |
| Grayscale | `img.convert('L')` | `cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)` |
| Blur | `img.filter(ImageFilter.GaussianBlur(r))` | `cv2.GaussianBlur(img, (k,k), sigma)` |
| Edge detection | Limited | `cv2.Canny(img, t1, t2)` |

**IMPORTANT:** OpenCV uses **BGR** channel order, matplotlib uses **RGB**.
Always convert: `img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)` before plotting.